# Explore the text before building classifiers

Exploratory data analysis (EDA) asks what the posts look like: what words
appear frequently, which words have high TF-IDF weights, and how long the
posts are. This notebook produces a word cloud, frequency bars, TF-IDF bars,
and a text-length histogram. It does not train a classifier.

**Reading route:** choose text → preserve originals → create analysis views
→ choose rows for plotting → draw each chart → inspect its underlying table.
Results describe the selected input rows; no real charts or counts are bundled.

## Connect to the shared project files

The `src/` folder is the shared toolbox. It holds the codebook and functions used
by multiple notebooks. The next cell finds it from the project root or `notebooks/`.
Keep the complete repository together. On Kaggle or Colab, extract or attach the
complete repository and set `PROJECT_ROOT` to its location when necessary.

Install the repository requirements before running the notebook. The setup cell
prints the root and label count so you can check the selected files. Data and
generated results remain in the ignored project folders.

In [ ]:
from pathlib import Path
import os
import sys

# Leave None when running from the repository root or notebooks/.
# On Kaggle/Colab, set this to the extracted repository directory when needed.
PROJECT_ROOT = os.environ.get("MINDMINER_PROJECT_ROOT")
if PROJECT_ROOT:
    REPO_ROOT = Path(PROJECT_ROOT).expanduser().resolve()
else:
    here = Path.cwd().resolve()
    REPO_ROOT = next((p for p in [here, *here.parents]
                      if (p / "src/mindminer_codebook.py").is_file()), None)
if REPO_ROOT is None or not (REPO_ROOT / "src/mindminer_codebook.py").is_file():
    raise FileNotFoundError("Extract the complete MindMiner repository and set PROJECT_ROOT to its folder.")
sys.path.insert(0, str(REPO_ROOT / "src"))
RUN_ROOT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else REPO_ROOT
from mindminer_codebook import CODEBOOK_VERSION, CODEBOOK_SHA256, CODEBOOK, FIELDS, TAGS, LABELS
print("Project root:", REPO_ROOT)
print("Codebook:", CODEBOOK_VERSION, "| tags:", len(LABELS))

In [ ]:
import pandas as pd
from IPython.display import display
from mindminer_text import preprocess_dataframe, visualize_text
from mindminer_utils import file_hash, save_json

FILE = Path(os.environ.get('MINDMINER_EDA_FILE', str(REPO_ROOT / 'data/raw/dataminds.csv')))
TEXT_COLUMN = None  # Or set 'text', 'text_original', or the actual column name.
TOP_N = 20
MAX_CLOUD_WORDS = 100
EXTRA_STOPWORDS = set()
FONT_PATH = None
DEDUPLICATE_FOR_PLOTS = True
ENABLE_LEMMATIZATION = False
USE_LEMMATIZED_FOR_PLOTS = False
OUTPUT = RUN_ROOT / 'results/eda'

## 1. Choose the text and preserve the original

A row is one post. The notebook reads CSV, TSV, or Excel and chooses
`text_original`, `text`, or `text_clean`, in that order, unless you set
`TEXT_COLUMN`. It keeps a copy of the source text for comparison.

We create two views. `text_clean` repairs encoding/HTML and whitespace while
retaining ordinary language, punctuation, and negation. `text_analysis` is
a lowercase view with URLs, email addresses, and user handles removed for
word summaries. Neither procedure establishes clinical meaning.

Text-quality flags use the `text_` prefix. They do not overwrite annotation
review flags. The source file is not modified by this notebook.

In [ ]:
if not FILE.is_file():
    raise FileNotFoundError(f'Set FILE to an existing input table: {FILE}')
if FILE.suffix.lower() in {'.xlsx', '.xlsm'}:
    raw_data = pd.read_excel(FILE, engine='openpyxl')
elif FILE.suffix.lower() in {'.csv', '.tsv'}:
    raw_data = pd.read_csv(FILE, sep='\t' if FILE.suffix.lower() == '.tsv' else ',', encoding='utf-8-sig')
else:
    raise ValueError('Use CSV, TSV, XLSX, or XLSM for this notebook.')
if TEXT_COLUMN is None:
    TEXT_COLUMN = next((c for c in ['text_original', 'text', 'text_clean'] if c in raw_data), None)
if TEXT_COLUMN not in raw_data:
    raise ValueError(f'Set TEXT_COLUMN to one of: {raw_data.columns.tolist()}')
data = raw_data.copy()
data['text_original'] = raw_data[TEXT_COLUMN].copy()
data = preprocess_dataframe(data, source='text_original')
display(data[['text_original', 'text_clean', 'text_analysis']].head())
display(pd.Series({
    'input_rows': len(data),
    'empty_cleaned_text': int(data['clean_is_empty'].sum()),
    'empty_analysis_text': int(data['analysis_is_empty'].sum()),
    'duplicate_cleaned_text_all_copies': int(data['is_duplicate_clean'].sum()),
    'text_rows_needing_review': int(data['text_needs_review'].sum()),
}, name='count').to_frame())

## 2. Optional lemmatization

A lemma is a base word form, such as `walk` for `walking`. Lemmatization
can combine related word forms for descriptive summaries, but it also
changes the representation. It is optional and disabled by default.

To use it, install `requirements-optional.txt`, download the spaCy English
model with `python -m spacy download en_core_web_sm`, then enable the setting.
Negation is protected. The resulting view is used in charts only when
`USE_LEMMATIZED_FOR_PLOTS=True`; classifier notebooks retain their own text
preprocessing rather than importing this EDA vocabulary.

In [ ]:
if USE_LEMMATIZED_FOR_PLOTS and not ENABLE_LEMMATIZATION:
    raise ValueError('Enable lemmatization before choosing its output for plots.')
if ENABLE_LEMMATIZATION:
    import spacy
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
    negations = {'no', 'not', 'nor', 'never', 'none', 'neither', 'cannot', "n't", 'without'}
    stopwords = set(ENGLISH_STOP_WORDS) - negations
    def keep_lemmas(doc):
        return ' '.join(token.text if token.lower_ in negations else token.lemma_.lower()
                        for token in doc if not token.is_space and not token.is_punct
                        and not token.like_num and (token.lower_ in negations or token.lower_ not in stopwords))
    data['text_lemmatized'] = [keep_lemmas(doc) for doc in nlp.pipe(data['text_analysis'].tolist())]
    display(data[['text_analysis', 'text_lemmatized']].head())

## 3. Decide which rows the charts describe

Repeated copies of a post can inflate word counts. By default, plotting
keeps the first exact `text_clean` copy and excludes empty analysis text.
This affects only the plotting table; it does not rewrite modeling splits.
The printed audit shows each removal. Near-duplicates and paraphrases are
not automatically detected.

A chart describes the rows supplied to it. If EDA guides model settings,
use development data for those decisions and record that choice.

In [ ]:
plot_column = 'text_lemmatized' if USE_LEMMATIZED_FOR_PLOTS else 'text_analysis'
viz_data = data.drop_duplicates(subset='text_clean', keep='first').copy() if DEDUPLICATE_FOR_PLOTS else data.copy()
duplicates_removed = len(data) - len(viz_data)
usable = viz_data[plot_column].map(lambda value: isinstance(value, str) and bool(value.strip()))
empty_excluded = int((~usable).sum())
viz_data = viz_data.loc[usable].copy()
text = viz_data[plot_column].copy()
print(f'Input rows: {len(data)}; duplicate copies removed: {duplicates_removed}; empty rows excluded: {empty_excluded}; plotted: {len(text)}')

## 4. Understand the four charts

| Chart | What it shows | How to read it |
| --- | --- | --- |
| Word cloud | Filtered individual-word occurrence counts | Larger words occurred more often; size is not a model importance score |
| Word-frequency bars | The most frequent retained words | Bar length is the number of occurrences across posts |
| TF-IDF keyword bars | Mean normalized TF-IDF weight for each word | A different summary from raw frequency, influenced by document frequency and post normalization |
| Text-length histogram | Word/number token counts per post before stopword removal | Each bin counts posts in a length range; the dashed line is the median |

English stopwords such as common function words are filtered from keyword
charts, while explicit negation is protected. Word/number tokens here are
not BERT subword tokens. The histogram therefore cannot directly measure
how many posts exceed a transformer's 512-token limit.

Each post remains a separate TF-IDF document. This EDA vectorizer is fitted
for description only and is never reused inside classification evaluation.

In [ ]:
plot_results = visualize_text(
    text, top_n=TOP_N, max_cloud_words=MAX_CLOUD_WORDS,
    extra_stopwords=EXTRA_STOPWORDS, output_dir=OUTPUT, font_path=FONT_PATH,
)

## 5. Look behind the pictures

The tables contain the exact values used to draw the charts. Raw count
and mean TF-IDF answer different questions, so their rankings can differ.
A frequent word is not automatically a risk factor, a causal explanation,
or a clinically validated indicator.

In [ ]:
display(plot_results['word_frequency'].head(TOP_N))
display(plot_results['tfidf_keywords'].head(TOP_N))
display(plot_results['text_length_summary'].to_frame())
save_json(OUTPUT / 'run_settings.json', {
    'input_sha256': file_hash(FILE), 'source_column': TEXT_COLUMN,
    'plot_column': plot_column, 'input_rows': len(data), 'plotted_rows': len(text),
    'deduplicate_for_plots': DEDUPLICATE_FOR_PLOTS,
    'duplicate_copies_removed': duplicates_removed, 'empty_rows_excluded': empty_excluded,
    'top_n': TOP_N, 'max_cloud_words': MAX_CLOUD_WORDS,
    'extra_stopwords': sorted(EXTRA_STOPWORDS),
    'codebook_version': CODEBOOK_VERSION, 'codebook_sha256': CODEBOOK_SHA256,
})
print('Saved figures, tables, and settings:', OUTPUT)